In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
import json

In [3]:
file_path = '../../data/raw/data-new.json'
with open(file_path, "r") as file:
    data =json.load(file)

In [4]:
tree = pd.DataFrame(data['nodes'])

In [5]:
tree.head()

,id,name,thesis,school,country,year,subject,advisors,students
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]"
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]"
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[]
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[]
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[]


In [6]:
unique_subjects = tree["subject"].dropna().unique()

In [7]:
unique_subjects

array(['74—Mechanics of deformable solids', '62—Statistics',
       '68—Computer science', '08—General algebraic systems',
       '03—Mathematical logic and foundations',
       '06—Order, lattices, ordered algebraic structures',
       '65—Numerical analysis',
       '90—Operations research, mathematical programming',
       '44—Integral transforms, operational calculus',
       '97—Mathematics education',
       '82—Statistical mechanics, structure of matter',
       '76—Fluid mechanics', '54—General topology',
       '52—Convex and discrete geometry', '46—Functional analysis',
       '60—Probability theory and stochastic processes',
       '37—Dynamical systems and ergodic theory',
       '57—Manifolds and cell complexes',
       '34—Ordinary differential equations', '11—Number theory',
       '94—Information and communication, circuits',
       '20—Group theory and generalizations',
       '16—Associative rings and algebras',
       '30—Functions of a complex variable',
       '41—

In [8]:
sorted_subs = sorted(unique_subjects, key=lambda x: int(x.split("—")[0]))

In [9]:
subject_codes = [s.split("—")[0] for s in sorted_subs]
subject_names = [s.split("—")[1] for s in sorted_subs]
subjects = pd.DataFrame({
    "subject_Code": subject_codes,
    "subject_Name": subject_names
})

In [10]:
subjects

,subject_Code,subject_Name
0,00,General
1,01,History and biography
2,03,Mathematical logic and foundations
3,05,Combinatorics
4,06,"Order, lattices, ordered algebraic structures"
...,...,...
58,91,"Game theory, economics, social and behavioral ..."
59,92,Biology and other natural sciences
60,93,Systems theory; control
61,94,"Information and communication, circuits"


## Checking if there are subject areas with slight changes 

In [11]:
import re

def normalize_subject_name(name):
    if pd.isna(name):
        return None
    
    name = name.lower()
    name = name.replace("&", "and")
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    return name

subjects["subject_Name_clean"] = subjects["subject_Name"].apply(normalize_subject_name)

In [12]:
subjects.head()

,subject_Code,subject_Name,subject_Name_clean
0,00,General,general
1,01,History and biography,history and biography
2,03,Mathematical logic and foundations,mathematical logic and foundations
3,05,Combinatorics,combinatorics
4,06,"Order, lattices, ordered algebraic structures",order lattices ordered algebraic structures


In [13]:
subjects[subjects.duplicated("subject_Name_clean", keep=False)].sort_values("subject_Name_clean")

,subject_Code,subject_Name,subject_Name_clean


In [14]:
!pip install rapidfuzz

In [15]:
from rapidfuzz import fuzz

pairs = []

names = subjects[["subject_Code", "subject_Name", "subject_Name_clean"]].dropna()

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        row1 = names.iloc[i]
        row2 = names.iloc[j]
        
        score = fuzz.token_sort_ratio(
            row1["subject_Name_clean"],
            row2["subject_Name_clean"]
        )
        
        if score >= 80:
            pairs.append({
                "code_1": row1["subject_Code"],
                "name_1": row1["subject_Name"],
                "code_2": row2["subject_Code"],
                "name_2": row2["subject_Name"],
                "similarity": score
            })

similar_subjects = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
similar_subjects

,code_1,name_1,code_2,name_2,similarity
1,16,Associative rings and algebras,17,Nonassociative rings and algebras,95.238095
3,34,Ordinary differential equations,35,Partial differential equations,85.245902
2,26,Real functions,33,Special functions,83.870968
0,13,Commutative rings and algebras,16,Associative rings and algebras,83.333333


In [16]:
tree["subject_code"] = tree["subject"].apply(lambda x: x.split("—")[0] if pd.notna(x) else None)
tree["subject_name"] = tree["subject"].apply(lambda x: x.split("—")[1] if pd.notna(x) else None)

In [17]:
tree.head(10)

,id,name,thesis,school,country,year,subject,advisors,students,subject_code,subject_name
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]",74,Mechanics of deformable solids
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]",None,None
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[],None,None
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[],74,Mechanics of deformable solids
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[],None,None
5,6,William B. Stiles,Solutions of Clamped Plated Problems by Means ...,Iowa State University,UnitedStates,1945.0,None,[258],[],None,None
6,9,Henry David Block,Explicit Solutions of Certain Singular Integra...,Iowa State University,UnitedStates,1949.0,None,[281],"[23, 28, 71, 101996, 87826]",None,None
7,8,James W. Beach,Flow of Viscous Fluid between Slowing Rotating...,Iowa State University,UnitedStates,1948.0,None,[258],[],None,None
8,7,Carl Eric Langenhop,Properties of Kernels of Integral Equations Wh...,Iowa State University,UnitedStates,1948.0,None,[281],"[48, 104689, 38, 33, 3026, 39, 32, 35]",None,None
9,10,Frank E. Bortle,Analytical Study of Dynamic Loads on Elastical...,Iowa State University,UnitedStates,1949.0,None,[258],[],None,None


# Missing subjects

In [18]:
# make sure that the advisor and student lists are actual lists
def to_list(x):
    if isinstance(x, list):
        return x
    
    if pd.isna(x):
        return []
    
    if isinstance(x, str):
        if x in ["", "[]" , "None"]:
            return []
        return x.strip()
        
    return []

tree["advisors"] = tree["advisors"].apply(to_list)
tree["students"] = tree["students"].apply(to_list)

In [19]:
tree.head()


,id,name,thesis,school,country,year,subject,advisors,students,subject_code,subject_name
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]",74,Mechanics of deformable solids
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]",None,None
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[],None,None
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[],74,Mechanics of deformable solids
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[],None,None


In [20]:
from collections import Counter 

In [21]:

def collect_subjects_from_neighbors(
    start_id,
    neighbor_dict,
    id_to_subject,
    max_depth=2
):   
    visited = set()
    queue = [(start_id, 0)]
    votes = Counter()
    
    while queue:
        current_id, depth = queue.pop(0)
        
        if depth >= max_depth:
            continue
        
        for neighbor_id in neighbor_dict.get(current_id, []):
            if neighbor_id in visited:
                continue
            
            visited.add(neighbor_id)
            
            subject = id_to_subject.get(neighbor_id)
            
            if subject is not None:
                # closer relatives get more weight
                weight = 1 / (depth + 1)
                votes[subject] += weight
            
            queue.append((neighbor_id, depth + 1))
    
    return votes

In [22]:
id_to_subject = dict(zip(tree["id"], tree["subject_code"]))
id_to_advisors = dict(zip(tree["id"], tree["advisors"]))
id_to_students = dict(zip(tree["id"], tree["students"]))

In [23]:
id_to_students.get(93643, [])

[302089,
 135405,
 135464,
 135502,
 135381,
 201153,
 135503,
 135442,
 239314,
 135391,
 135383,
 211193,
 259071,
 306425,
 135456,
 135384,
 135500,
 135808,
 254084,
 135501,
 154380,
 135809,
 135403,
 138006,
 165252,
 165255,
 141813,
 135392,
 152108,
 135436,
 135508,
 135471,
 144081,
 135462,
 271000,
 135393,
 135431,
 306424,
 196625,
 225060,
 184397,
 296518,
 135465,
 135506,
 174684,
 135390,
 198292,
 135402,
 212966,
 135451,
 224114,
 135386,
 135449,
 201154,
 294673,
 196623,
 174683,
 165253,
 135509,
 144078,
 38445,
 135430,
 135445,
 135505,
 135454,
 191119,
 135512,
 135448,
 135438,
 135443,
 148004,
 135472,
 135444,
 144079,
 135401,
 135475,
 201152,
 135399,
 141815,
 135511,
 135400,
 135385,
 309958,
 211192,
 135474,
 211194,
 235504,
 135453,
 161686,
 135478,
 135398,
 135507,
 271831,
 196624,
 135504,
 135450,
 184399,
 135452,
 184398,
 157688,
 154379,
 322860,
 322862,
 135437,
 135513,
 135387,
 212967,
 135409,
 135433,
 135408,
 135429,
 1

In [24]:
collect_subjects_from_neighbors(
    start_id=93643,
    neighbor_dict=id_to_students,
    id_to_subject=id_to_subject,
    max_depth=2
)

Counter({'94': 126.5, '68': 49.5, '93': 1.0, '20': 0.5})

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [26]:
known = tree[
    tree["subject_code"].notna() &
    tree["thesis"].notna()
].copy()

title_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

title_model.fit(known["thesis"], known["subject_code"])

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(min_df=2, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

In [27]:
subject_classes = title_model.named_steps["clf"].classes_

def thesis_subject_votes(thesis, top_k=5):
    if pd.isna(thesis) or thesis is None:
        return Counter()
    
    probs = title_model.predict_proba([thesis])[0]
    
    top_indices = probs.argsort()[::-1][:top_k]
    
    votes = Counter()
    for i in top_indices:
        votes[subject_classes[i]] = probs[i]
    
    return votes

In [28]:
def infer_subject_for_id(
    person_id,
    thesis,
    id_to_subject,
    id_to_advisors,
    id_to_students,
    max_depth=2,
    w_descendants=0.45,
    w_ancestors=0.25,
    w_thesis=0.30
):
    descendant_votes = collect_subjects_from_neighbors(
        person_id,
        id_to_students,
        id_to_subject,
        max_depth=max_depth
    )
    
    ancestor_votes = collect_subjects_from_neighbors(
        person_id,
        id_to_advisors,
        id_to_subject,
        max_depth=max_depth
    )
    
    thesis_votes = thesis_subject_votes(thesis)
    
    total_votes = Counter()
    
    for code, score in descendant_votes.items():
        total_votes[code] += w_descendants * score
    
    for code, score in ancestor_votes.items():
        total_votes[code] += w_ancestors * score
    
    for code, score in thesis_votes.items():
        total_votes[code] += w_thesis * score
    
    if not total_votes:
        return None, 0, {}
    
    best_code, best_score = total_votes.most_common(1)[0]
    
    total_score = sum(total_votes.values())
    confidence = best_score / total_score if total_score > 0 else 0
    
    return best_code, confidence, dict(total_votes)

In [29]:
missing_mask = tree["subject_code"].isna()

results = tree.loc[missing_mask].apply(
    lambda row: infer_subject_for_id(
        person_id=row["id"],
        thesis=row["thesis"],
        id_to_subject=id_to_subject,
        id_to_advisors=id_to_advisors,
        id_to_students=id_to_students,
        max_depth=2
    ),
    axis=1
)

tree.loc[missing_mask, "predicted_subject_code"] = results.apply(lambda x: x[0])
tree.loc[missing_mask, "subject_prediction_confidence"] = results.apply(lambda x: x[1])
tree.loc[missing_mask, "subject_prediction_votes"] = results.apply(lambda x: x[2])

In [30]:
tree.head()

,id,name,thesis,school,country,year,subject,advisors,students,subject_code,subject_name,predicted_subject_code,subject_prediction_confidence,subject_prediction_votes
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]",74,Mechanics of deformable solids,NaN,NaN,NaN
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]",None,None,81,0.389770,"{'81': 0.125, '34': 0.10889493712225978, '41':..."
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[],None,None,74,0.469316,"{'74': 0.05825367916239288, '91': 0.0350892674..."
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[],74,Mechanics of deformable solids,NaN,NaN,NaN
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[],None,None,74,0.976849,"{'74': 0.26153942054761814, '70': 0.0025129365..."


In [31]:
subjects.loc[subjects["subject_Code"] == "47"]

,subject_Code,subject_Name,subject_Name_clean
34,47,Operator theory,operator theory


In [32]:
tree["final_subject_code"] = tree["subject_code"]

tree.loc[
    tree["final_subject_code"].isna(),
    "final_subject_code"
] = tree.loc[
    tree["final_subject_code"].isna(),
    "predicted_subject_code"
]

In [33]:
tree


,id,name,thesis,school,country,year,subject,advisors,students,subject_code,subject_name,predicted_subject_code,subject_prediction_confidence,subject_prediction_votes,final_subject_code
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]",74,Mechanics of deformable solids,NaN,NaN,NaN,74
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]",None,None,81,0.389770,"{'81': 0.125, '34': 0.10889493712225978, '41':...",81
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[],None,None,74,0.469316,"{'74': 0.05825367916239288, '91': 0.0350892674...",74
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[],74,Mechanics of deformable solids,NaN,NaN,NaN,74
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[],None,None,74,0.976849,"{'74': 0.26153942054761814, '70': 0.0025129365...",74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338527,346122,Maria Heuss,Responsible Advice-Giving Systems: Fairness an...,Universiteit van Amsterdam,Netherlands,2025.0,68—Computer science,"[229284, 84191]",[],68,Computer science,NaN,NaN,NaN,68
338528,346123,Konstantin Burlachenko,Optimization Methods and Software for Federate...,King Abdullah University of Science and Techno...,SaudiArabia,2025.0,"90—Operations research, mathematical programming",[11470],[],90,"Operations research, mathematical programming",NaN,NaN,NaN,90
338529,346119,Jacob Stream Laxer,Packable Weak Poincare Inequality and Uniform ...,State University of New York at Stony Brook,UnitedStates,2025.0,28—Measure and integration,[18456],[],28,Measure and integration,NaN,NaN,NaN,28
338530,29316,Detlef Dürr,Wahrscheinlichste Wege von Diffusionsprozessen,Universität Münster,Germany,1978.0,60—Probability theory and stochastic processes,[300879],"[49602, 191494, 273114, 28892, 28895, 171502, ...",60,Probability theory and stochastic processes,NaN,NaN,NaN,60


In [34]:
tree = tree.merge(
    subjects,
    left_on="final_subject_code",
    right_on="subject_Code",
    how="left",
    suffixes=("", "_from_table")
)

In [35]:
tree.head()

,id,name,thesis,school,country,year,subject,advisors,students,subject_code,subject_name,predicted_subject_code,subject_prediction_confidence,subject_prediction_votes,final_subject_code,subject_Code,subject_Name,subject_Name_clean
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]",74,Mechanics of deformable solids,NaN,NaN,NaN,74,74,Mechanics of deformable solids,mechanics of deformable solids
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]",None,None,81,0.389770,"{'81': 0.125, '34': 0.10889493712225978, '41':...",81,81,Quantum Theory,quantum theory
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[],None,None,74,0.469316,"{'74': 0.05825367916239288, '91': 0.0350892674...",74,74,Mechanics of deformable solids,mechanics of deformable solids
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[],74,Mechanics of deformable solids,NaN,NaN,NaN,74,74,Mechanics of deformable solids,mechanics of deformable solids
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[],None,None,74,0.976849,"{'74': 0.26153942054761814, '70': 0.0025129365...",74,74,Mechanics of deformable solids,mechanics of deformable solids


In [36]:
tree[[
    "id",
    "name",
    "thesis",
    "subject",
    "final_subject_code",
    "subject_Name",
    "subject_prediction_confidence"
]].head()

,id,name,thesis,subject,final_subject_code,subject_Name,subject_prediction_confidence
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,74—Mechanics of deformable solids,74,Mechanics of deformable solids,NaN
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,None,81,Quantum Theory,0.389770
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,None,74,Mechanics of deformable solids,0.469316
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,74—Mechanics of deformable solids,74,Mechanics of deformable solids,NaN
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,None,74,Mechanics of deformable solids,0.976849


In [37]:
tree.to_csv('../../data/processed/tree_with_subjects.csv', index=False)